# MarkProbe-Core: image embedding and attack inference

This notebook loads a trained checkpoint, embeds a text payload into one input image, applies several attack conditions, and saves the original image, attacked image, decoded bits, decoded text, and confidence data for every condition.

In [ ]:
from pathlib import Path
import json
import math
import random
import sys

import numpy as np
import torch
from PIL import Image, ImageDraw, ImageOps
from IPython.display import display, Markdown

PROJECT = Path.cwd().resolve()
if not (PROJECT / 'markprobe_core').is_dir():
    PROJECT = PROJECT.parent
if not (PROJECT / 'markprobe_core').is_dir():
    raise FileNotFoundError('Open this notebook from the MarkProbe-Core project or set PROJECT manually.')
sys.path.insert(0, str(PROJECT))

from markprobe_core import (
    attack, bits_to_text, crop_resize_attack, decode_image,
    jpeg_compress_decompress, load_checkpoint, quantize_8bit_ste, text_to_bits,
)

DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
CHECKPOINT = PROJECT / 'runs' / 'markprobe_core.pt'
INPUT_IMAGE = PROJECT / 'data' / 'input.png'
OUTPUT_DIR = PROJECT / 'runs' / 'inference_attacks'
TEXT = 'MarkProbe test payload'
RANDOM_SEED = 2026

if not CHECKPOINT.is_file():
    raise FileNotFoundError(f'Checkpoint not found: {CHECKPOINT}')
if not INPUT_IMAGE.is_file():
    raise FileNotFoundError(f'Input image not found: {INPUT_IMAGE}. Set INPUT_IMAGE to an image file.')

model, DEVICE = load_checkpoint(CHECKPOINT, device=DEVICE)
MESSAGE_BITS = model.message_bits
payload = text_to_bits(TEXT, MESSAGE_BITS)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Model: {CHECKPOINT}')
print(f'Device: {DEVICE} | image size: {model.image_size} | message bits: {MESSAGE_BITS}')
print(f'Input: {INPUT_IMAGE}')
print(f'Output directory: {OUTPUT_DIR}')

In [ ]:
def pil_to_tensor(image, image_size):
    image = ImageOps.fit(image.convert('RGB'), (image_size, image_size), method=Image.Resampling.BICUBIC)
    array = np.asarray(image, dtype=np.float32) / 255.0
    return torch.from_numpy(array).permute(2, 0, 1).unsqueeze(0)

def tensor_to_pil(tensor):
    array = tensor.detach().clamp(0, 1)[0].permute(1, 2, 0).cpu().numpy()
    return Image.fromarray(np.uint8(np.round(array * 255.0)), 'RGB')

def display_pair(original, attacked, title):
    original = original.copy()
    attacked = attacked.copy()
    original.thumbnail((512, 512))
    attacked.thumbnail((512, 512))
    label_height = 32
    canvas = Image.new('RGB', (original.width + attacked.width, max(original.height, attacked.height) + label_height), 'white')
    draw = ImageDraw.Draw(canvas)
    draw.text((8, 8), 'Original', fill='black')
    draw.text((original.width + 8, 8), title, fill='black')
    canvas.paste(original, (0, label_height))
    canvas.paste(attacked, (original.width, label_height))
    display(canvas)

cover = Image.open(INPUT_IMAGE).convert('RGB')
cover_tensor = pil_to_tensor(cover, model.image_size).to(DEVICE)
payload_tensor = torch.tensor(payload, dtype=torch.float32, device=DEVICE).view(1, -1)
with torch.no_grad():
    encoded_tensor = model.encode(cover_tensor, payload_tensor).clamp(0, 1)
encoded_tensor = quantize_8bit_ste(encoded_tensor).detach()
encoded = tensor_to_pil(encoded_tensor)
print(f'Payload text: {TEXT!r}')
print(f'Encoded image size: {encoded.size}')

In [ ]:
# Each entry returns the image that will actually be saved and decoded.
def make_views(encoded_tensor):
    views = {}
    views['clean_png'] = quantize_8bit_ste(encoded_tensor)
    views['jpeg_quality_75'] = jpeg_compress_decompress(views['clean_png'], quality=75, chroma_subsampling=True)
    random.seed(RANDOM_SEED + 10)
    torch.manual_seed(RANDOM_SEED + 10)
    views['crop_80_95'] = crop_resize_attack(encoded_tensor, 0.80, 0.95)
    random.seed(RANDOM_SEED + 20)
    torch.manual_seed(RANDOM_SEED + 20)
    views['photo_attack'] = attack(encoded_tensor, strength=1.0)
    return {name: value.clamp(0, 1).detach() for name, value in views.items()}

views = make_views(encoded_tensor)
print('Attack conditions:', ', '.join(views))

In [ ]:
results = {}
for name, attacked_tensor in views.items():
    attack_dir = OUTPUT_DIR / name
    attack_dir.mkdir(parents=True, exist_ok=True)
    original_path = attack_dir / 'original.png'
    attacked_path = attack_dir / 'attacked.png'
    cover.save(original_path)
    tensor_to_pil(attacked_tensor).save(attacked_path)

    decoded_bits, probabilities = decode_image(model, attacked_path, DEVICE)
    correct = sum(expected == actual for expected, actual in zip(payload, decoded_bits))
    try:
        decoded_text = bits_to_text(decoded_bits)
    except (ValueError, UnicodeDecodeError):
        decoded_text = None
    result = {
        'attack': name,
        'input_text': TEXT,
        'message_bits': payload,
        'decoded_bits': decoded_bits,
        'probabilities': probabilities,
        'decoded_text': decoded_text,
        'bit_accuracy': correct / len(payload),
        'ber': 1.0 - correct / len(payload),
        'original': str(original_path),
        'attacked': str(attacked_path),
    }
    (attack_dir / 'decoded.json').write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding='utf-8')
    (attack_dir / 'decoded.txt').write_text(decoded_text or '<invalid text payload>', encoding='utf-8')
    results[name] = result

    display(Markdown(f"## {name}\n\nDecoded text: `{decoded_text if decoded_text is not None else '<invalid text payload>'}`  \nBit accuracy: `{result['bit_accuracy']:.2%}` | BER: `{result['ber']:.2%}`"))
    display_pair(cover, tensor_to_pil(attacked_tensor), name)

(OUTPUT_DIR / 'summary.json').write_text(json.dumps(results, ensure_ascii=False, indent=2), encoding='utf-8')
print(f'Saved inference artifacts to {OUTPUT_DIR}')